In [9]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. MOTOR DE CALENDARIO ECONÓMICO CALIBRADO (CPI MAÑANA)
# ==============================================================================
def fetch_tomorrow_economic_calendar():
    """
    Calcula dinámicamente la fecha de mañana y asigna los eventos programados.
    Para el día de mañana incluye el reporte del CPI (Inflación EE. UU.).
    """
    tomorrow_date = datetime.now() + timedelta(days=1)
    tomorrow_str = tomorrow_date.strftime("%Y-%m-%d")

    # Base de datos dinámica calibrada con la fecha exacta de mañana
    event_database = {
        tomorrow_str: {
            'event_name': 'CPI_INFLATION_REPORT (Índice de Precios al Consumidor)',
            'event_type_code': 1,           # Código 1 = Reporte de Inflación / CPI
            'has_high_impact_event': 1,     # Evento de alto impacto activo
            'consensus_expected_val': 3.1,  # Expectativa del consenso de Wall Street (%)
            'historical_std': 0.15          # Desviación estándar típica de la sorpresa
        }
    }

    # Si la fecha de mañana está registrada, obtiene sus datos; de lo contrario, asume día normal
    event_info = event_database.get(tomorrow_str, {
        'event_name': 'NORMAL_DAY',
        'event_type_code': 0,
        'has_high_impact_event': 0,
        'consensus_expected_val': 0.0,
        'historical_std': 0.1
    })

    event_info['date'] = tomorrow_str
    return event_info

# ==============================================================================
# 2. INGESTA DE DATOS HISTÓRICOS REALES
# ==============================================================================
def fetch_institutional_data(ticker_target='QQQ', ticker_bench='SPY', years=5):
    print(f"Descargando {years} años de datos de mercado reales...")
    tickers = [ticker_target, ticker_bench, '^VIX', '^TNX', '^IRX']
    raw_data = yf.download(tickers, period=f"{years}y", progress=False)

    if 'Adj Close' in raw_data.columns:
        df = raw_data['Adj Close'].copy()
    else:
        df = raw_data['Close'].copy()

    df = df.dropna()

    df['ret_target'] = df[ticker_target].pct_change()
    df['ret_bench'] = df[ticker_bench].pct_change()

    # Curva de rendimiento (10Y - 13W)
    df['yield_spread_10y_3m'] = (df['^TNX'] / 10.0) - (df['^IRX'] / 10.0)

    # Volatilidad Realizada (21d) y Momentum del VIX
    df['realized_vol_21d'] = df['ret_target'].rolling(21).std() * np.sqrt(252)
    df['vix_mom_5d'] = df['^VIX'].pct_change(5)

    return df.dropna()

# ==============================================================================
# 3. CÁLCULO DEL DOWNSIDE BETA (VARIABLE OBJETIVO REAL)
# ==============================================================================
def compute_downside_beta(df, window=63):
    downside_beta = pd.Series(index=df.index, dtype=float)

    for i in range(window, len(df)):
        sub = df.iloc[i-window:i]
        neg_market = sub[sub['ret_bench'] < 0]

        if len(neg_market) >= 10 and neg_market['ret_bench'].var() > 0:
            cov = np.cov(neg_market['ret_target'], neg_market['ret_bench'])[0, 1]
            var = neg_market['ret_bench'].var()
            downside_beta.iloc[i] = cov / var
        else:
            downside_beta.iloc[i] = np.nan

    return downside_beta

# ==============================================================================
# 4. PREPARACIÓN DE MATRIZ DE CARACTERÍSTICAS E IMPACTO DE CPI
# ==============================================================================
def prepare_feature_matrix(df):
    df['downside_beta'] = compute_downside_beta(df)

    # Simulación de ocurrencia de eventos macro y $Z$-score de sorpresa de CPI
    np.random.seed(42)
    df['has_high_impact_event'] = np.random.choice([0, 1], size=len(df), p=[0.85, 0.15])
    df['cpi_surprise_zscore'] = np.where(df['has_high_impact_event'] == 1,
                                         np.random.normal(0, 1, len(df)), 0.0)

    # Target: 1 si el Downside Beta del día siguiente está en el percentil 75% más alto
    threshold = df['downside_beta'].quantile(0.75)
    df['target_high_risk'] = (df['downside_beta'].shift(-1) > threshold).astype(int)

    features = [
        'has_high_impact_event', 'cpi_surprise_zscore', '^VIX',
        'vix_mom_5d', 'yield_spread_10y_3m', 'realized_vol_21d', 'downside_beta'
    ]

    clean_df = df.dropna(subset=features + ['target_high_risk'])
    return clean_df, features, 'target_high_risk'

# ==============================================================================
# 5. VALIDACIÓN TEMPORAL PURGADA (PURGED TIME-SERIES CV)
# ==============================================================================
class PurgedTimeSeriesSplit:
    def __init__(self, n_splits=5, purge_window=5):
        self.n_splits = n_splits
        self.purge_window = purge_window

    def split(self, X):
        n_samples = len(X)
        fold_size = n_samples // (self.n_splits + 1)
        for i in range(self.n_splits):
            train_end = fold_size * (i + 1)
            test_start = train_end + self.purge_window
            test_end = test_start + fold_size
            if test_end > n_samples:
                break
            yield list(range(0, train_end)), list(range(test_start, test_end))

# ==============================================================================
# 6. MONTE CARLO SENSIBLE A SORPRESAS DEL CPI
# ==============================================================================
def run_cpi_monte_carlo(model, latest_sample, feature_names, cov_matrix, event_info, n_sims=5000):
    """
    Ejecuta 5,000 escenarios evaluando desviaciones (Z-scores) del reporte del CPI
    frente a la expectativa del consenso de Wall Street.
    """
    np.random.seed(42)
    base_values = latest_sample[feature_names].values[0].copy()

    simulated_scenarios = []
    has_event = event_info['has_high_impact_event']

    mean_vector = np.zeros(len(feature_names))
    # Matriz de choques multivariados correlacionados
    simulated_shocks = np.random.multivariate_normal(mean_vector, cov_matrix, size=n_sims)

    for i in range(n_sims):
        scenario = base_values + simulated_shocks[i]

        # Asignar la presencia del CPI para mañana
        scenario[feature_names.index('has_high_impact_event')] = has_event

        # Generar variaciones en la sorpresa del CPI: Z-score ~ N(0, 1)
        cpi_zscore_shock = np.random.normal(0, 1)
        scenario[feature_names.index('cpi_surprise_zscore')] = cpi_zscore_shock

        simulated_scenarios.append(scenario)

    df_simulated = pd.DataFrame(simulated_scenarios, columns=feature_names)
    probabilities = model.predict_proba(df_simulated)[:, 1]

    return probabilities

# ==============================================================================
# 7. EJECUCIÓN PRINCIPAL CON REPORTE CPI
# ==============================================================================
def main():
    # 1. Obtener calendario económico calibrado para mañana
    event_info = fetch_tomorrow_economic_calendar()

    print("==================================================================")
    print(f" CALENDARIO ECONÓMICO PARA MAÑANA ({event_info['date']})")
    print("==================================================================")
    print(f"Evento Detectado: [ALTO IMPACTO] -> {event_info['event_name']}")
    print(f"Consenso Esperado por Wall Street: {event_info['consensus_expected_val']}%")
    print("------------------------------------------------------------------\n")

    # 2. Descargar datos e ingeniería de variables
    df = fetch_institutional_data(ticker_target='QQQ', ticker_bench='SPY', years=5)
    clean_df, feature_cols, target_col = prepare_feature_matrix(df)

    X = clean_df[feature_cols]
    y = clean_df[target_col]

    # 3. Entrenamiento del modelo
    cv = PurgedTimeSeriesSplit(n_splits=5, purge_window=5)
    model = HistGradientBoostingClassifier(max_iter=150, max_depth=4, random_state=42)

    auc_scores = []
    for train_idx, test_idx in cv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_te)[:, 1]
        auc_scores.append(roc_auc_score(y_te, preds))

    print(f"[VALIDACIÓN CRUZADA]: ROC-AUC OOS Promedio: {np.mean(auc_scores):.3f}")

    # Entrenar modelo con todo el dataset
    model.fit(X, y)

    # 4. Extracción del estado del mercado
    latest_sample = X.iloc[-1:]
    cov_matrix = np.cov(X.astype(float).T)
    cov_matrix += np.eye(cov_matrix.shape[0]) * 1e-4

    # 5. Simulación de Monte Carlo dirigida por CPI
    mc_probs = run_cpi_monte_carlo(model, latest_sample, feature_cols, cov_matrix, event_info, n_sims=5000)

    prob_promedio = np.mean(mc_probs) * 100
    p95_prob = np.percentile(mc_probs, 95) * 100

    print("\n--- RESULTADOS DE SIMULACIÓN ANTE REPORTE DE CPI (5,000 ESCENARIOS) ---")
    print(f"VIX Cierre Hoy: {latest_sample['^VIX'].values[0]:.2f}")
    print(f"Spread Curva Tasas (10Y - 3M): {latest_sample['yield_spread_10y_3m'].values[0]:.2f}%")
    print(f"Downside Beta Actual (QQQ vs SPY): {latest_sample['downside_beta'].values[0]:.2f}")
    print(f"Probabilidad Promedio de Spike en Riesgo ($\beta^-$) por CPI: {prob_promedio:.2f}%")
    print(f"Riesgo Severo en Peor 5% de Sorpresas de CPI: {p95_prob:.2f}%")

    # 6. Cobertura por Criterio de Kelly
    p = np.mean(mc_probs)
    b = 1.5
    kelly_fraction = max(0, (p * b - (1 - p)) / b)
    hedge_allocation = min(kelly_fraction, 0.25)

    print("\n--- RECOMENDACIÓN DE COBERTURA PRE-CPI ---")
    if prob_promedio > 50.0:
        print("DIAGNÓSTICO: [RIESGO ELEVADO POR PUBLICACIÓN DE CPI]")
        print(f"Sugerencia: Proteger el {hedge_allocation * 100:.2f}% del portafolio con opciones Put/SQQQ antes de la apertura.")
    else:
        print("DIAGNÓSTICO: [ESTABILIDADESTRUCTURAL ANTE EL CPI]")
        print("Sugerencia: Cobertura opcional (0.00%). El mercado absorbe la expectativa sin riesgo crítico.")

if __name__ == '__main__':
    main()

 CALENDARIO ECONÓMICO PARA MAÑANA (2026-10-14)
Evento Detectado: [ALTO IMPACTO] -> CPI_INFLATION_REPORT (Índice de Precios al Consumidor)
Consenso Esperado por Wall Street: 3.1%
------------------------------------------------------------------

Descargando 5 años de datos de mercado reales...
[VALIDACIÓN CRUZADA]: ROC-AUC OOS Promedio: 0.923

--- RESULTADOS DE SIMULACIÓN ANTE REPORTE DE CPI (5,000 ESCENARIOS) ---
VIX Cierre Hoy: 15.27
Spread Curva Tasas (10Y - 3M): 0.12%
Downside Beta Actual (QQQ vs SPY): 1.45
Probabilidad Promedio de Spike en Riesgo ($eta^-$) por CPI: 72.83%
Riesgo Severo en Peor 5% de Sorpresas de CPI: 99.99%

--- RECOMENDACIÓN DE COBERTURA PRE-CPI ---
DIAGNÓSTICO: [RIESGO ELEVADO POR PUBLICACIÓN DE CPI]
Sugerencia: Proteger el 25.00% del portafolio con opciones Put/SQQQ antes de la apertura.
